In [1]:
from roboflow import Roboflow
rf = Roboflow(api_key="aQMyLnAv8R2wkKBNsKGC")
project = rf.workspace("augmented-startups").project("drowsiness-detection-cntmz")
version = project.version(1)
dataset = version.download("multiclass")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Drowsiness-Detection-1 in multiclass:: 100%|██████████| 534/534 [00:04<00:00, 113.34it/s]


In [2]:
import os
import shutil

def merge_val_test_dirs(dataset_path):
    val_dir = os.path.join(dataset_path, 'valid')
    test_dir = os.path.join(dataset_path, 'test')
    
    if not os.path.exists(val_dir) or not os.path.exists(test_dir):
        print("Either 'valid' or 'test' directory doesn't exist.")
        return
    
    for item in os.listdir(test_dir):
        s = os.path.join(test_dir, item)
        d = os.path.join(val_dir, item)
        if os.path.isfile(s):
            shutil.move(s, d)
        elif os.path.isdir(s):
            shutil.move(s, os.path.join(val_dir, os.path.basename(s)))
    
    os.rmdir(test_dir)
    
    print("Merged 'test' into 'valid' directory. 'test' directory has been removed.")

dataset_path = 'Drowsiness-Detection-1'

merge_val_test_dirs(dataset_path)

subdirs = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]
print(f"Current subdirectories: {subdirs}")

val_dir = os.path.join(dataset_path, 'valid')
val_files = [f for f in os.listdir(val_dir) if os.path.isfile(os.path.join(val_dir, f))]
print(f"Number of files in 'valid' directory: {len(val_files)}")

Either 'valid' or 'test' directory doesn't exist.
Current subdirectories: ['train', 'valid']
Number of files in 'valid' directory: 175


In [3]:
import os
from PIL import Image
import json

dataset_path = './Drowsiness-Detection-1'

def explore_dataset(dataset_path):
    subdirs = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]
    
    for subdir in subdirs:
        subdir_path = os.path.join(dataset_path, subdir)
        image_files = [f for f in os.listdir(subdir_path) if f.endswith(('.jpg', '.jpeg', '.png'))]
        
        print(f"\nExploring {subdir} set:")
        print(f"Number of images: {len(image_files)}")
        
        if image_files:
            first_image_path = os.path.join(subdir_path, image_files[0])
            with Image.open(first_image_path) as img:
                print(f"Image dimensions (first image): {img.size}")
        
    classes_path = os.path.join(dataset_path, 'classes.txt')
    if os.path.exists(classes_path):
        with open(classes_path, 'r') as f:
            classes = f.read().splitlines()
        print(f"\nClasses in the dataset: {classes}")

explore_dataset(dataset_path)


Exploring train set:
Number of images: 352
Image dimensions (first image): (1920, 1080)

Exploring valid set:
Number of images: 174
Image dimensions (first image): (1920, 1080)


In [4]:
import torch
import numpy as np
from torchvision import transforms,datasets
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import random_split,DataLoader

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


In [10]:
import os

def list_subdirectories(path):
    print(f"Contents of {path}:")
    for item in os.listdir(path):
        item_path = os.path.join(path, item)
        if os.path.isdir(item_path):
            print(f"  - {item}/")
        else:
            print(f"  - {item}")

list_subdirectories('Drowsiness-Detection-1')

Contents of Drowsiness-Detection-1:
  - README.dataset.txt
  - README.roboflow.txt
  - train/
  - valid/


In [18]:
#train_dataset=datasets.ImageFolder(root='Drowsiness-Detection-1/train')
#test_dataset=datasets.ImageFolder(root='Drowsiness-Detection-1/valid')

FileNotFoundError: Couldn't find any class folder in Drowsiness-Detection-1/train/.

In [19]:
train_transform=transforms.Compose([
    transforms.Resize((145,145)),
    transforms.RandomHorizontalFlip(p=0.25),
    transforms.RandomRotation(60),
    transforms.ToTensor(),
    transforms.Normalize((0,0,0),(1,1,1))
])

test_transform=transforms.Compose([
    transforms.Resize((145,145)),
    transforms.ToTensor(),
    transforms.Normalize((0,0,0),(1,1,1))
])

In [20]:
full_dataset=datasets.ImageFolder(root='Drowsiness-Detection-1')

In [21]:
train_size=int(0.8*len(full_dataset))
test_size=len(full_dataset)-train_size

In [22]:
train_dataset,test_dataset=random_split(full_dataset,[train_size,test_size])

In [23]:
train_dataset.dataset.transform = train_transform
test_dataset.dataset.transform = test_transform

In [24]:
train_loader=DataLoader(train_dataset,batch_size=32)
test_loader=DataLoader(test_dataset,batch_size=32)

In [25]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1=nn.Conv2d(in_channels=3,out_channels=512,kernel_size=3)
        self.conv2=nn.Conv2d(in_channels=512,out_channels=256,kernel_size=3)
        self.conv3=nn.Conv2d(in_channels=256,out_channels=64,kernel_size=3)
        self.conv4=nn.Conv2d(in_channels=64,out_channels=32,kernel_size=3)

        self.pool=nn.MaxPool2d(kernel_size=2)
        self.relu=nn.ReLU()
        self.adaptive_pool = nn.AdaptiveAvgPool2d((6, 6))

        self.fc1=nn.Linear(in_features=6*6*32,out_features=64)
        self.fc2=nn.Linear(in_features=64,out_features=1)

    def forward(self,x):
        x=self.relu(self.conv1(x))
        x=self.pool(x)
        x=self.relu(self.conv2(x))
        x=self.pool(x)
        x=self.relu(self.conv3(x))
        x=self.pool(x)
        x=self.relu(self.conv4(x))
        x=self.pool(x)
        x=self.adaptive_pool(x)
        x=x.view(x.size(0),-1)
        x=self.relu(self.fc1(x))
        x=self.fc2(x)
        return x

In [26]:
model=CNN()
criterion = nn.BCEWithLogitsLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.01)
epoch=16

In [27]:
from tqdm import tqdm

train_losses = []
test_losses = []
accuracy_list = []

for epochs in range(epoch):
    model.train()
    running_loss=0
    for images,labels in tqdm(train_loader):
        labels=labels.unsqueeze(1)
        labels=labels.float()
        outputs = model(images)
        loss=criterion(outputs, labels)
        running_loss+=loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    train_losses.append(running_loss / len(train_loader))

    model.eval()
    running_loss=0
    with torch.no_grad():
        for images,labels in tqdm(test_loader):
            labels=labels.unsqueeze(1)
            labels=labels.float()
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            preds = [1 if outputs[i] >= 0.5 else 0 for i in range(len(outputs))]
            acc = [1 if preds[i] == labels[i] else 0 for i in range(len(outputs))]
    
    acc = (np.sum(acc) / len(preds))*100
    test_losses.append(running_loss / len(test_loader))
    accuracy_list.append(acc)

    print(f"[Epoch: {epochs+1}/{epoch}], [Train loss: {train_losses[-1]:.4f}], [Val loss: {test_losses[-1]:.4f}], [Acc: {acc:.2f}%]")

100%|██████████| 4/4 [00:14<00:00,  3.65s/it]


[Epoch: 1/16], [Train loss: 10.3786], [Val loss: 0.6992], [Acc: 60.00%]


  0%|          | 0/14 [00:10<?, ?it/s]


KeyboardInterrupt: 